# 03 · Train Model B (temporal) on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bileldh-ing/PrintGuard-/blob/main/notebooks/03_train_model_b_colab.ipynb)

**Runtime → Change runtime type → GPU.**

Model B is a causal TCN / small Transformer over per-frame features (Model A outputs + a frozen
MobileNetV3-Small embedding). Steps: clone → install → data → Model A weights (train them, or upload
the zip from notebook 02) → ONNX export → feature cache (resumable; synthetic gradual layer shift is
generated here) → train TCN and Transformer → evaluate A vs B (frame AP, detection delay, false alarms)
→ CPU benchmark → download.

**Watch out for the scene shortcut:** stringing and clean YOLO-Strings sequences are different prints,
so a very high stringing AP for Model B may reflect print identity rather than stringing.

In [ ]:
!nvidia-smi || echo 'No GPU - switch the runtime to GPU'

In [ ]:
# Clone the repo (skipped when the notebook already runs inside it)
import os, subprocess, sys
if not os.path.exists("printguard/__init__.py"):
    if not os.path.exists("PrintGuard-"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bileldh-ing/PrintGuard-.git"], check=True)
    os.chdir("PrintGuard-")
print(os.getcwd())

In [ ]:
PROFILE = "full"    # "smoke" for a quick end-to-end check
DEVICE = "0"
SMOKE = PROFILE == "smoke"
WDIR = "weights/smoke" if SMOKE else "weights"
RDIR = "results/smoke" if SMOKE else "results"
OVR = "--override configs/smoke/model_b.yaml" if SMOKE else ""
!pip -q install -r requirements-train.txt
!make data PROFILE={PROFILE}

## Model A weights
Upload `model_a_artifacts.zip` from notebook 02, or train Model A here.

In [ ]:
import os
if os.path.exists("model_a_artifacts.zip"):
    !unzip -oq model_a_artifacts.zip
elif not os.path.exists(f"{WDIR}/model_a_detect.pt"):
    try:
        from google.colab import files
        up = files.upload()   # pick model_a_artifacts.zip, or cancel to train here
        !unzip -oq model_a_artifacts.zip
    except Exception:
        pass
if not os.path.exists(f"{WDIR}/model_a_detect.pt"):
    !make train-a PROFILE={PROFILE} DEVICE={DEVICE}

## ONNX export + feature cache (resumable)

In [ ]:
!make export PROFILE={PROFILE}
# Feature extraction is the expensive step; it caches one .npz per sequence and can be re-run to resume.
!python -m printguard.training.train_b features --config configs/model_b_tcn.yaml {OVR}

## Train TCN and Transformer

In [ ]:
!python -m printguard.training.train_b train --config configs/model_b_tcn.yaml {OVR}
!python -m printguard.training.train_b train --config configs/model_b_transformer.yaml {OVR}

## Evaluate Model B vs Model A on held-out test sequences

In [ ]:
!python -m printguard.training.train_b evaluate --config configs/model_b_tcn.yaml {OVR}
!python -m printguard.training.train_b evaluate --config configs/model_b_transformer.yaml {OVR}
from IPython.display import Markdown, display
for m in ["model_b_tcn", "model_b_transformer"]:
    display(Markdown(open(f"{RDIR}/{m}/metrics_test.md").read()))

## Optional: export the TCN to ONNX and benchmark on CPU

In [ ]:
!python -m printguard.export temporal --weights {WDIR}/model_b_tcn.pt --window 32
!make benchmark PROFILE={PROFILE}
import json; print(json.dumps(json.load(open(f"{RDIR}/benchmark_cpu.json"))["pipeline_per_frame"], indent=2))

## Download

In [ ]:
!zip -qr model_b_artifacts.zip {WDIR}/model_b_*.pt {WDIR}/*.onnx {RDIR}
try:
    from google.colab import files
    files.download("model_b_artifacts.zip")
except ImportError:
    print("saved model_b_artifacts.zip")

Report results with: profile, GPU, epochs, commit hash, and the alarm rule (default 3-of-5 above 0.5).
Update the README results table only from these generated files.